<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model05_Bureau_Balance_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:

DATA_PATH = "/content/drive/MyDrive/datasets/raw/"
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [3]:

application = pd.read_csv(DATA_PATH + "application_train.csv")
prev = pd.read_csv(DATA_PATH + "previous_application.csv")
bureau = pd.read_csv(DATA_PATH + "bureau.csv")
bureau_balance = pd.read_csv(DATA_PATH + "bureau_balance.csv")

print("Application shape:", application.shape)
print("Previous application shape:", prev.shape)
print("Bureau shape:", bureau.shape)
print("Bureau balance shape:", bureau_balance.shape)

Application shape: (307511, 122)
Previous application shape: (1670214, 37)
Bureau shape: (1716428, 17)
Bureau balance shape: (27299925, 3)


In [4]:
# Recreate MODEL02 application-level features
application["DAYS_EMPLOYED_ANOM"] = (application["DAYS_EMPLOYED"] == 365243).astype(int)
application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, np.nan)

application["EXT_SOURCE_1_MISSING"] = application["EXT_SOURCE_1"].isna().astype(int)
application["EXT_SOURCE_3_MISSING"] = application["EXT_SOURCE_3"].isna().astype(int)

application["AGE_YEARS"] = -application["DAYS_BIRTH"] / 365.25
application["EMPLOYMENT_YEARS"] = -application["DAYS_EMPLOYED"] / 365.25

application["CREDIT_INCOME_RATIO"] = application["AMT_CREDIT"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_INCOME_RATIO"] = application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_CREDIT_RATIO"] = application["AMT_ANNUITY"] / application["AMT_CREDIT"]
application["GOODS_CREDIT_RATIO"] = application["AMT_GOODS_PRICE"] / application["AMT_CREDIT"]
application["EMPLOYMENT_AGE_RATIO"] = application["EMPLOYMENT_YEARS"] / application["AGE_YEARS"]

ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
application["EXT_SOURCE_MEAN"] = application[ext_cols].mean(axis=1)
application["EXT_SOURCE_MIN"] = application[ext_cols].min(axis=1)
application["EXT_SOURCE_MAX"] = application[ext_cols].max(axis=1)
application["EXT_SOURCE_STD"] = application[ext_cols].std(axis=1)

application["EXT_SOURCE_1_2"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_2"]
application["EXT_SOURCE_1_3"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_3"]
application["EXT_SOURCE_2_3"] = application["EXT_SOURCE_2"] * application["EXT_SOURCE_3"]

print("MODEL02 application features recreated.")

MODEL02 application features recreated.


In [5]:
# Recreate MODEL03 previous-application features
prev_count = (
    prev.groupby("SK_ID_CURR")
    .size()
    .rename("PREV_APPLICATION_COUNT")
    .reset_index()
)

financial_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT": ["mean", "max", "sum"],
        "AMT_APPLICATION": ["mean", "max", "sum"],
        "AMT_ANNUITY": ["mean", "max", "sum"],
        "AMT_GOODS_PRICE": ["mean", "max", "sum"],
        "AMT_DOWN_PAYMENT": ["mean", "max"],
        "RATE_DOWN_PAYMENT": ["mean", "max"]
    })
)
financial_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in financial_agg.columns]
financial_agg = financial_agg.reset_index()

prev["PREV_CREDIT_APPL_RATIO"] = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"].replace(0, np.nan)
prev["PREV_CREDIT_APPL_DIFF"] = prev["AMT_CREDIT"] - prev["AMT_APPLICATION"]

relationship_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_CREDIT_APPL_RATIO": ["mean", "max"],
        "PREV_CREDIT_APPL_DIFF": ["mean", "max"]
    })
)
relationship_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in relationship_agg.columns]
relationship_agg = relationship_agg.reset_index()

prev["PREV_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
prev["PREV_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev["PREV_CANCELED"] = (prev["NAME_CONTRACT_STATUS"] == "Canceled").astype(int)
prev["PREV_UNUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Unused offer").astype(int)

status_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_APPROVED": "sum",
        "PREV_REFUSED": "sum",
        "PREV_CANCELED": "sum",
        "PREV_UNUSED": "sum"
    })
    .reset_index()
)
status_agg = status_agg.rename(columns={
    "PREV_APPROVED": "PREV_APPROVED_COUNT",
    "PREV_REFUSED": "PREV_REFUSED_COUNT",
    "PREV_CANCELED": "PREV_CANCELED_COUNT",
    "PREV_UNUSED": "PREV_UNUSED_COUNT"
})

status_agg = status_agg.merge(
    prev_count[["SK_ID_CURR", "PREV_APPLICATION_COUNT"]],
    on="SK_ID_CURR", how="left"
)
status_agg["PREV_APPROVAL_RATE"] = status_agg["PREV_APPROVED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_REFUSAL_RATE"] = status_agg["PREV_REFUSED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_CANCELLATION_RATE"] = status_agg["PREV_CANCELED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg = status_agg.drop(columns=["PREV_APPLICATION_COUNT"])

decision_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"DAYS_DECISION": ["min", "max", "mean"]})
)
decision_agg.columns = ["PREV_DAYS_DECISION_" + col[1].upper() for col in decision_agg.columns]
decision_agg = decision_agg.reset_index()

payment_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"CNT_PAYMENT": ["mean", "max", "sum"]})
)
payment_agg.columns = ["PREV_CNT_PAYMENT_" + col[1].upper() for col in payment_agg.columns]
payment_agg = payment_agg.reset_index()

prev_features = prev_count.copy()
prev_features = prev_features.merge(financial_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(relationship_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(status_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(decision_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(payment_agg, on="SK_ID_CURR", how="left")

print("Previous-application feature table:", prev_features.shape)

Previous-application feature table: (338857, 35)


In [6]:
# Recreate MODEL04 bureau-level features
bureau_count = (
    bureau.groupby("SK_ID_CURR")
    .size()
    .rename("BUREAU_CREDIT_COUNT")
    .reset_index()
)

bureau_financial_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT_SUM": ["mean", "max", "sum"],
        "AMT_CREDIT_SUM_DEBT": ["mean", "max", "sum"],
        "AMT_CREDIT_SUM_LIMIT": ["mean", "max"],
        "AMT_ANNUITY": ["mean"]
    })
)
bureau_financial_agg.columns = ["BUREAU_" + col[0] + "_" + col[1].upper() for col in bureau_financial_agg.columns]
bureau_financial_agg = bureau_financial_agg.reset_index()

bureau["BUREAU_OVERDUE_FLAG"] = (bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int)

bureau_overdue_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "CREDIT_DAY_OVERDUE": ["max"],
        "BUREAU_OVERDUE_FLAG": ["sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["max", "sum"]
    })
    .reset_index()
)
bureau_overdue_agg.columns = [
    "SK_ID_CURR", "BUREAU_OVERDUE_DAYS_MAX", "BUREAU_OVERDUE_COUNT",
    "BUREAU_OVERDUE_AMOUNT_MAX", "BUREAU_OVERDUE_AMOUNT_SUM"
]

bureau_overdue_agg = bureau_overdue_agg.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)
bureau_overdue_agg["BUREAU_OVERDUE_RATIO"] = (
    bureau_overdue_agg["BUREAU_OVERDUE_COUNT"] / bureau_overdue_agg["BUREAU_CREDIT_COUNT"]
)
bureau_overdue_agg = bureau_overdue_agg.drop(columns=["BUREAU_CREDIT_COUNT"])

bureau["BUREAU_ACTIVE_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau["BUREAU_CLOSED_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Closed").astype(int)
bureau["BUREAU_SOLD_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Sold").astype(int)
bureau["BUREAU_BAD_DEBT_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Bad debt").astype(int)

bureau_status_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "BUREAU_ACTIVE_FLAG": "sum",
        "BUREAU_CLOSED_FLAG": "sum",
        "BUREAU_SOLD_FLAG": "sum",
        "BUREAU_BAD_DEBT_FLAG": "sum"
    })
    .reset_index()
)
bureau_status_agg = bureau_status_agg.rename(columns={
    "BUREAU_ACTIVE_FLAG": "BUREAU_ACTIVE_COUNT",
    "BUREAU_CLOSED_FLAG": "BUREAU_CLOSED_COUNT",
    "BUREAU_SOLD_FLAG": "BUREAU_SOLD_COUNT",
    "BUREAU_BAD_DEBT_FLAG": "BUREAU_BAD_DEBT_COUNT"
})

bureau_status_agg = bureau_status_agg.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)
bureau_status_agg["BUREAU_ACTIVE_RATIO"] = (
    bureau_status_agg["BUREAU_ACTIVE_COUNT"] / bureau_status_agg["BUREAU_CREDIT_COUNT"]
)
bureau_status_agg = bureau_status_agg.drop(columns=["BUREAU_CREDIT_COUNT"])

bureau_type_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({"CREDIT_TYPE": "nunique"})
    .reset_index()
)
bureau_type_agg = bureau_type_agg.rename(columns={"CREDIT_TYPE": "BUREAU_CREDIT_TYPE_COUNT"})

bureau["BUREAU_CREDIT_CARD_FLAG"] = (bureau["CREDIT_TYPE"] == "Credit card").astype(int)
bureau["BUREAU_MORTGAGE_FLAG"] = (bureau["CREDIT_TYPE"] == "Mortgage").astype(int)
bureau["BUREAU_MICROLOAN_FLAG"] = (bureau["CREDIT_TYPE"] == "Microloan").astype(int)

bureau_type_specific = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "BUREAU_CREDIT_CARD_FLAG": "sum",
        "BUREAU_MORTGAGE_FLAG": "sum",
        "BUREAU_MICROLOAN_FLAG": "sum"
    })
    .reset_index()
)
bureau_type_specific = bureau_type_specific.rename(columns={
    "BUREAU_CREDIT_CARD_FLAG": "BUREAU_CREDIT_CARD_COUNT",
    "BUREAU_MORTGAGE_FLAG": "BUREAU_MORTGAGE_COUNT",
    "BUREAU_MICROLOAN_FLAG": "BUREAU_MICROLOAN_COUNT"
})

bureau_type_agg = bureau_type_agg.merge(bureau_type_specific, on="SK_ID_CURR", how="left")

bureau_timing_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "DAYS_CREDIT": ["min", "max", "mean"],
        "DAYS_CREDIT_UPDATE": ["mean"]
    })
)
bureau_timing_agg.columns = ["BUREAU_" + col[0] + "_" + col[1].upper() for col in bureau_timing_agg.columns]
bureau_timing_agg = bureau_timing_agg.reset_index()

bureau_prolong_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({"CNT_CREDIT_PROLONG": "sum"})
    .reset_index()
)
bureau_prolong_agg = bureau_prolong_agg.rename(columns={"CNT_CREDIT_PROLONG": "BUREAU_CREDIT_PROLONG_TOTAL"})

bureau_features = bureau_count.copy()
bureau_features = bureau_features.merge(bureau_financial_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_overdue_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_status_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_type_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_timing_agg, on="SK_ID_CURR", how="left")
bureau_features = bureau_features.merge(bureau_prolong_agg, on="SK_ID_CURR", how="left")

bureau_feature_cols = [col for col in bureau_features.columns if col != "SK_ID_CURR"]

print("Bureau feature table:", bureau_features.shape)
print("Bureau features:", len(bureau_feature_cols))

Bureau feature table: (305811, 30)
Bureau features: 29


In [7]:
# Basic bureau_balance structure check
print("Bureau balance columns:", bureau_balance.columns.tolist())
print("\nUnique SK_ID_BUREAU:", bureau_balance["SK_ID_BUREAU"].nunique())
print("Rows:", len(bureau_balance))
print("\nStatus distribution:")
display(bureau_balance["STATUS"].value_counts(dropna=False))

Bureau balance columns: ['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS']

Unique SK_ID_BUREAU: 817395
Rows: 27299925

Status distribution:


,count
STATUS,
C,13646993
0,7499507
X,5810482
1,242347
5,62406
2,23419
3,8924
4,5847


In [8]:
months_per_bureau = bureau_balance.groupby("SK_ID_BUREAU").size()

print("Monthly records per bureau credit:")
display(months_per_bureau.describe())

Monthly records per bureau credit:


,0
count,817395.000000
mean,33.398693
std,25.794666
min,1.000000
25%,13.000000
50%,26.000000
75%,48.000000
max,97.000000


In [18]:
# 9 — Check MONTHS_BALANCE range
print("MONTHS_BALANCE distribution:")
display(bureau_balance["MONTHS_BALANCE"].describe())

MONTHS_BALANCE distribution:


,MONTHS_BALANCE
count,2.729992e+07
mean,-3.074169e+01
std,2.386451e+01
min,-9.600000e+01
25%,-4.600000e+01
50%,-2.500000e+01
75%,-1.100000e+01
max,0.000000e+00


In [19]:
# Create monthly status flags (0=no DPD, 1=mild, 2-5=severe, C=closed, X=unknown)
bureau_balance["BB_STATUS_0"] = (bureau_balance["STATUS"] == "0").astype(int)
bureau_balance["BB_STATUS_1"] = (bureau_balance["STATUS"] == "1").astype(int)
bureau_balance["BB_STATUS_2_PLUS"] = bureau_balance["STATUS"].isin(["2", "3", "4", "5"]).astype(int)
bureau_balance["BB_STATUS_C"] = (bureau_balance["STATUS"] == "C").astype(int)
bureau_balance["BB_STATUS_X"] = (bureau_balance["STATUS"] == "X").astype(int)

In [20]:
#Aggregate monthly history to SK_ID_BUREAU (stage 1 of 2)
bureau_balance_credit_agg = (
    bureau_balance.groupby("SK_ID_BUREAU")
    .agg({
        "MONTHS_BALANCE": ["min", "max", "count"],
        "BB_STATUS_0": "sum",
        "BB_STATUS_1": "sum",
        "BB_STATUS_2_PLUS": "sum",
        "BB_STATUS_C": "sum",
        "BB_STATUS_X": "sum"
    })
)
bureau_balance_credit_agg.columns = ["BB_" + col[0] + "_" + col[1].upper() for col in bureau_balance_credit_agg.columns]
bureau_balance_credit_agg = bureau_balance_credit_agg.reset_index()

display(bureau_balance_credit_agg.head())

,SK_ID_BUREAU,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_COUNT,BB_BB_STATUS_0_SUM,BB_BB_STATUS_1_SUM,BB_BB_STATUS_2_PLUS_SUM,BB_BB_STATUS_C_SUM,BB_BB_STATUS_X_SUM
0,5001709,-96,0,97,0,0,0,86,11
1,5001710,-82,0,83,5,0,0,48,30
2,5001711,-3,0,4,3,0,0,0,1
3,5001712,-18,0,19,10,0,0,9,0
4,5001713,-21,0,22,0,0,0,0,22


In [21]:
#Delinquency count and rate at credit level
bureau_balance_credit_agg["BB_DELINQUENCY_COUNT"] = (
    bureau_balance_credit_agg["BB_BB_STATUS_1_SUM"] + bureau_balance_credit_agg["BB_BB_STATUS_2_PLUS_SUM"]
)
bureau_balance_credit_agg["BB_DELINQUENCY_RATE"] = (
    bureau_balance_credit_agg["BB_DELINQUENCY_COUNT"] / bureau_balance_credit_agg["BB_MONTHS_BALANCE_COUNT"]
)
bureau_balance_credit_agg["BB_STATUS_2_PLUS_RATE"] = (
    bureau_balance_credit_agg["BB_BB_STATUS_2_PLUS_SUM"] / bureau_balance_credit_agg["BB_MONTHS_BALANCE_COUNT"]
)

display(bureau_balance_credit_agg.head())

,SK_ID_BUREAU,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_COUNT,BB_BB_STATUS_0_SUM,BB_BB_STATUS_1_SUM,BB_BB_STATUS_2_PLUS_SUM,BB_BB_STATUS_C_SUM,BB_BB_STATUS_X_SUM,BB_DELINQUENCY_COUNT,BB_DELINQUENCY_RATE,BB_STATUS_2_PLUS_RATE
0,5001709,-96,0,97,0,0,0,86,11,0,0.0,0.0
1,5001710,-82,0,83,5,0,0,48,30,0,0.0,0.0
2,5001711,-3,0,4,3,0,0,0,1,0,0.0,0.0
3,5001712,-18,0,19,10,0,0,9,0,0,0.0,0.0
4,5001713,-21,0,22,0,0,0,0,22,0,0.0,0.0


In [22]:
#Clean up double-prefixed column names from the aggregation
rename_map = {
    "BB_BB_STATUS_0_SUM": "BB_STATUS_0_COUNT",
    "BB_BB_STATUS_1_SUM": "BB_STATUS_1_COUNT",
    "BB_BB_STATUS_2_PLUS_SUM": "BB_STATUS_2_PLUS_COUNT",
    "BB_BB_STATUS_C_SUM": "BB_STATUS_C_COUNT",
    "BB_BB_STATUS_X_SUM": "BB_STATUS_X_COUNT"
}
bureau_balance_credit_agg = bureau_balance_credit_agg.rename(columns=rename_map)

display(bureau_balance_credit_agg.head())

,SK_ID_BUREAU,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_COUNT,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_PLUS_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT,BB_DELINQUENCY_COUNT,BB_DELINQUENCY_RATE,BB_STATUS_2_PLUS_RATE
0,5001709,-96,0,97,0,0,0,86,11,0,0.0,0.0
1,5001710,-82,0,83,5,0,0,48,30,0,0.0,0.0
2,5001711,-3,0,4,3,0,0,0,1,0,0.0,0.0
3,5001712,-18,0,19,10,0,0,9,0,0,0.0,0.0
4,5001713,-21,0,22,0,0,0,0,22,0,0.0,0.0


In [23]:
# Ever-delinquent flags at credit level
bureau_balance_credit_agg["BB_EVER_DELINQUENT"] = (bureau_balance_credit_agg["BB_DELINQUENCY_COUNT"] > 0).astype(int)
bureau_balance_credit_agg["BB_EVER_SEVERE_DELINQUENCY"] = (bureau_balance_credit_agg["BB_STATUS_2_PLUS_COUNT"] > 0).astype(int)

display(bureau_balance_credit_agg.head())

,SK_ID_BUREAU,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_COUNT,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_PLUS_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT,BB_DELINQUENCY_COUNT,BB_DELINQUENCY_RATE,BB_STATUS_2_PLUS_RATE,BB_EVER_DELINQUENT,BB_EVER_SEVERE_DELINQUENCY
0,5001709,-96,0,97,0,0,0,86,11,0,0.0,0.0,0,0
1,5001710,-82,0,83,5,0,0,48,30,0,0.0,0.0,0,0
2,5001711,-3,0,4,3,0,0,0,1,0,0.0,0.0,0,0
3,5001712,-18,0,19,10,0,0,9,0,0,0.0,0.0,0,0
4,5001713,-21,0,22,0,0,0,0,22,0,0.0,0.0,0,0


In [24]:
# Merge credit-level features onto bureau (brings in SK_ID_CURR)
bureau_with_balance = bureau[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(
    bureau_balance_credit_agg, on="SK_ID_BUREAU", how="left"
)

print("Bureau + balance shape:", bureau_with_balance.shape)
display(bureau_with_balance.head())

Bureau + balance shape: (1716428, 15)


,SK_ID_CURR,SK_ID_BUREAU,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_COUNT,BB_STATUS_0_COUNT,BB_STATUS_1_COUNT,BB_STATUS_2_PLUS_COUNT,BB_STATUS_C_COUNT,BB_STATUS_X_COUNT,BB_DELINQUENCY_COUNT,BB_DELINQUENCY_RATE,BB_STATUS_2_PLUS_RATE,BB_EVER_DELINQUENT,BB_EVER_SEVERE_DELINQUENCY
0,215354,5714462,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
#Verify credit-level merge integrity
print("Unique SK_ID_BUREAU:", bureau_with_balance["SK_ID_BUREAU"].nunique())
print("Duplicate SK_ID_BUREAU:", bureau_with_balance["SK_ID_BUREAU"].duplicated().sum())

Unique SK_ID_BUREAU: 1716428
Duplicate SK_ID_BUREAU: 0


In [26]:
# Aggregate bureau-balance features to applicant level (stage 2 of 2)
# Note: this aggregates ACROSS all of an applicant's bureau credits,
bureau_balance_app_agg = (
    bureau_with_balance.groupby("SK_ID_CURR")
    .agg({
        "BB_MONTHS_BALANCE_MIN": ["min", "max"],
        "BB_MONTHS_BALANCE_MAX": ["min", "max"],
        "BB_MONTHS_BALANCE_COUNT": ["mean", "max", "sum"],
        "BB_DELINQUENCY_COUNT": ["mean", "max", "sum"],
        "BB_DELINQUENCY_RATE": ["mean", "max"],
        "BB_STATUS_2_PLUS_RATE": ["mean", "max"],
        "BB_STATUS_1_COUNT": ["sum"],
        "BB_STATUS_2_PLUS_COUNT": ["sum"],
        "BB_STATUS_C_COUNT": ["sum"],
        "BB_STATUS_X_COUNT": ["sum"],
        "BB_EVER_DELINQUENT": ["sum"],
        "BB_EVER_SEVERE_DELINQUENCY": ["sum"]
    })
)
bureau_balance_app_agg.columns = ["BB_" + col[0] + "_" + col[1].upper() for col in bureau_balance_app_agg.columns]
bureau_balance_app_agg = bureau_balance_app_agg.reset_index()

# Clean up double "BB_BB_" prefixes
bureau_balance_app_agg.columns = [col.replace("BB_BB_", "BB_") for col in bureau_balance_app_agg.columns]

display(bureau_balance_app_agg.head())

,SK_ID_CURR,BB_MONTHS_BALANCE_MIN_MIN,BB_MONTHS_BALANCE_MIN_MAX,BB_MONTHS_BALANCE_MAX_MIN,BB_MONTHS_BALANCE_MAX_MAX,BB_MONTHS_BALANCE_COUNT_MEAN,BB_MONTHS_BALANCE_COUNT_MAX,BB_MONTHS_BALANCE_COUNT_SUM,BB_DELINQUENCY_COUNT_MEAN,BB_DELINQUENCY_COUNT_MAX,...,BB_DELINQUENCY_RATE_MEAN,BB_DELINQUENCY_RATE_MAX,BB_STATUS_2_PLUS_RATE_MEAN,BB_STATUS_2_PLUS_RATE_MAX,BB_STATUS_1_COUNT_SUM,BB_STATUS_2_PLUS_COUNT_SUM,BB_STATUS_C_COUNT_SUM,BB_STATUS_X_COUNT_SUM,BB_EVER_DELINQUENT_SUM,BB_EVER_SEVERE_DELINQUENCY_SUM
0,100001,-51.0,-1.0,0.0,0.0,24.571429,52.0,172.0,0.142857,1.0,...,0.007519,0.052632,0.0,0.0,1.0,0.0,110.0,30.0,1.0,0.0
1,100002,-47.0,-3.0,-32.0,0.0,13.750000,22.0,110.0,3.375000,6.0,...,0.255682,0.500000,0.0,0.0,27.0,0.0,23.0,15.0,6.0,0.0
2,100003,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
3,100004,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
4,100005,-12.0,-2.0,0.0,0.0,7.000000,13.0,21.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,5.0,2.0,0.0,0.0


In [27]:
# Number of bureau credits with monthly history, and coverage ratio (cleaned, dead code removed)
credits_with_balance = (
    bureau_with_balance.groupby("SK_ID_CURR")["BB_MONTHS_BALANCE_COUNT"]
    .count()
    .rename("BUREAU_CREDITS_WITH_BALANCE_HISTORY")
    .reset_index()
)

credits_with_balance = credits_with_balance.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR",
    how="left"
)

credits_with_balance["BUREAU_BALANCE_COVERAGE_RATIO"] = (
    credits_with_balance["BUREAU_CREDITS_WITH_BALANCE_HISTORY"]
    / credits_with_balance["BUREAU_CREDIT_COUNT"]
)

credits_with_balance = credits_with_balance.drop(columns=["BUREAU_CREDIT_COUNT"])

display(credits_with_balance.head())

,SK_ID_CURR,BUREAU_CREDITS_WITH_BALANCE_HISTORY,BUREAU_BALANCE_COVERAGE_RATIO
0,100001,7,1.0
1,100002,8,1.0
2,100003,0,0.0
3,100004,0,0.0
4,100005,3,1.0


In [28]:
# Merge all bureau-balance applicant-level features together
bureau_balance_features = bureau_balance_app_agg.copy()
bureau_balance_features = bureau_balance_features.merge(credits_with_balance, on="SK_ID_CURR", how="left")

print("Bureau-balance applicant feature table:", bureau_balance_features.shape)
print("Unique applicants:", bureau_balance_features["SK_ID_CURR"].nunique())
print("Duplicate applicants:", bureau_balance_features["SK_ID_CURR"].duplicated().sum())

display(bureau_balance_features.head())

Bureau-balance applicant feature table: (305811, 23)
Unique applicants: 305811
Duplicate applicants: 0


,SK_ID_CURR,BB_MONTHS_BALANCE_MIN_MIN,BB_MONTHS_BALANCE_MIN_MAX,BB_MONTHS_BALANCE_MAX_MIN,BB_MONTHS_BALANCE_MAX_MAX,BB_MONTHS_BALANCE_COUNT_MEAN,BB_MONTHS_BALANCE_COUNT_MAX,BB_MONTHS_BALANCE_COUNT_SUM,BB_DELINQUENCY_COUNT_MEAN,BB_DELINQUENCY_COUNT_MAX,...,BB_STATUS_2_PLUS_RATE_MEAN,BB_STATUS_2_PLUS_RATE_MAX,BB_STATUS_1_COUNT_SUM,BB_STATUS_2_PLUS_COUNT_SUM,BB_STATUS_C_COUNT_SUM,BB_STATUS_X_COUNT_SUM,BB_EVER_DELINQUENT_SUM,BB_EVER_SEVERE_DELINQUENCY_SUM,BUREAU_CREDITS_WITH_BALANCE_HISTORY,BUREAU_BALANCE_COVERAGE_RATIO
0,100001,-51.0,-1.0,0.0,0.0,24.571429,52.0,172.0,0.142857,1.0,...,0.0,0.0,1.0,0.0,110.0,30.0,1.0,0.0,7,1.0
1,100002,-47.0,-3.0,-32.0,0.0,13.750000,22.0,110.0,3.375000,6.0,...,0.0,0.0,27.0,0.0,23.0,15.0,6.0,0.0,8,1.0
2,100003,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
3,100004,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
4,100005,-12.0,-2.0,0.0,0.0,7.000000,13.0,21.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,5.0,2.0,0.0,0.0,3,1.0


In [29]:
bureau_balance_feature_cols = [col for col in bureau_balance_features.columns if col != "SK_ID_CURR"]

print("Number of bureau-balance features:", len(bureau_balance_feature_cols))
for col in bureau_balance_feature_cols:
    print("-", col)

Number of bureau-balance features: 22
- BB_MONTHS_BALANCE_MIN_MIN
- BB_MONTHS_BALANCE_MIN_MAX
- BB_MONTHS_BALANCE_MAX_MIN
- BB_MONTHS_BALANCE_MAX_MAX
- BB_MONTHS_BALANCE_COUNT_MEAN
- BB_MONTHS_BALANCE_COUNT_MAX
- BB_MONTHS_BALANCE_COUNT_SUM
- BB_DELINQUENCY_COUNT_MEAN
- BB_DELINQUENCY_COUNT_MAX
- BB_DELINQUENCY_COUNT_SUM
- BB_DELINQUENCY_RATE_MEAN
- BB_DELINQUENCY_RATE_MAX
- BB_STATUS_2_PLUS_RATE_MEAN
- BB_STATUS_2_PLUS_RATE_MAX
- BB_STATUS_1_COUNT_SUM
- BB_STATUS_2_PLUS_COUNT_SUM
- BB_STATUS_C_COUNT_SUM
- BB_STATUS_X_COUNT_SUM
- BB_EVER_DELINQUENT_SUM
- BB_EVER_SEVERE_DELINQUENCY_SUM
- BUREAU_CREDITS_WITH_BALANCE_HISTORY
- BUREAU_BALANCE_COVERAGE_RATIO


In [30]:
#Build MODEL04 dataset (application + previous_application + bureau)
application_model = application.copy()
application_model = application_model.merge(prev_features, on="SK_ID_CURR", how="left")
application_model = application_model.merge(bureau_features, on="SK_ID_CURR", how="left")

print("Shape after Model04 features:", application_model.shape)

Shape after Model04 features: (307511, 202)


In [31]:
#Add bureau-balance features on top
application_model = application_model.merge(bureau_balance_features, on="SK_ID_CURR", how="left")

print("Shape after Bureau Balance features:", application_model.shape)
print("Unique applicants:", application_model["SK_ID_CURR"].nunique())
print("Duplicate applicants:", application_model["SK_ID_CURR"].duplicated().sum())

Shape after Bureau Balance features: (307511, 224)
Unique applicants: 307511
Duplicate applicants: 0


In [32]:
#Bureau-balance history flag (single source of truth, created post-merge)
application_model["HAS_BUREAU_BALANCE_HISTORY"] = application_model["BUREAU_CREDITS_WITH_BALANCE_HISTORY"].notna().astype(int)

print(application_model["HAS_BUREAU_BALANCE_HISTORY"].value_counts())

HAS_BUREAU_BALANCE_HISTORY
1    263491
0     44020
Name: count, dtype: int64


In [33]:
X = application_model.drop(columns=["TARGET", "SK_ID_CURR"])
y = application_model["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 223)
y shape: (307511,)


In [34]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

Training shape: (246008, 223)
Validation shape: (61503, 223)

Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Validation target distribution:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [35]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 207
Categorical features: 16


In [36]:
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [37]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [38]:
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

In [39]:
print("Training XGBoost with Model04 + Bureau Balance features...")
xgb_pipeline.fit(X_train, y_train)
print("Training complete.")

Training XGBoost with Model04 + Bureau Balance features...
Training complete.


In [40]:
valid_proba = xgb_pipeline.predict_proba(X_valid)[:, 1]
print("Predictions generated.")

Predictions generated.


In [41]:
roc_auc = roc_auc_score(y_valid, valid_proba)
pr_auc = average_precision_score(y_valid, valid_proba)

print("XGBOOST + MODEL04 + BUREAU BALANCE")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

XGBOOST + MODEL04 + BUREAU BALANCE
ROC-AUC: 0.7777
PR-AUC:  0.2745


In [42]:
MODEL04_ROC_AUC = 0.777585
MODEL04_PR_AUC = 0.274161

roc_change = roc_auc - MODEL04_ROC_AUC
pr_change = pr_auc - MODEL04_PR_AUC

print("IMPROVEMENT OVER MODEL04")
print(f"ROC-AUC change: {roc_change:+.4f}")
print(f"PR-AUC change:  {pr_change:+.4f}")

IMPROVEMENT OVER MODEL04
ROC-AUC change: +0.0001
PR-AUC change:  +0.0004


In [43]:
bureau_balance_result = pd.DataFrame({
    "Experiment": ["XGBoost + Model04 + Bureau Balance Features"],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
    "ROC-AUC Change": [roc_change],
    "PR-AUC Change": [pr_change]
})

display(bureau_balance_result.style.format({
    "ROC-AUC": "{:.4f}",
    "PR-AUC": "{:.4f}",
    "ROC-AUC Change": "{:+.4f}",
    "PR-AUC Change": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change
0,XGBoost + Model04 + Bureau Balance Features,0.7777,0.2745,+0.0001,+0.0004


In [44]:
bureau_balance_result.to_csv(RESULTS_PATH + "bureau_balance_experiment.csv", index=False)
print("Experiment saved to:", RESULTS_PATH + "bureau_balance_experiment.csv")

Experiment saved to: /content/drive/MyDrive/RupeeRisk/bureau_balance_experiment.csv


In [45]:
!pip install mlflow -q
import mlflow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [46]:
mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")
with mlflow.start_run(run_name="XGBoost_Bureau_Balance"):
    mlflow.log_param("stage", "Model05 - Bureau Balance Feature Engineering")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("builds_on", "Model04 application + previous application + bureau features")
    mlflow.log_param("n_bureau_balance_features", len(bureau_balance_feature_cols))
    mlflow.log_param("scale_pos_weight", False)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc_change_vs_Model04", roc_change)
    mlflow.log_metric("pr_auc_change_vs_Model04", pr_change)

print("Model05 logged to MLflow.")

<Experiment: artifact_location='/content/mlruns/1', creation_time=1787393296537, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787393296537, lifecycle_stage='active', name='RupeeRisk', tags={}, trace_location=None, workspace='default'>

In [47]:
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Bureau,0.777585,0.274161
1,XGBoost_Previous_Application,0.775428,0.265853
2,XGBoost_Application_Features,0.769403,0.262725
3,XGBoost_scale_pos_weight,0.760000,0.249300
4,XGBoost_Baseline,0.761200,0.251600
5,Logistic_Regression_Baseline,0.750100,0.232600


In [48]:
print("Bureau Balance features created:")
for col in bureau_balance_feature_cols:
    print("-", col)
print("\nTotal Bureau Balance features:", len(bureau_balance_feature_cols))

Bureau Balance features created:
- BB_MONTHS_BALANCE_MIN_MIN
- BB_MONTHS_BALANCE_MIN_MAX
- BB_MONTHS_BALANCE_MAX_MIN
- BB_MONTHS_BALANCE_MAX_MAX
- BB_MONTHS_BALANCE_COUNT_MEAN
- BB_MONTHS_BALANCE_COUNT_MAX
- BB_MONTHS_BALANCE_COUNT_SUM
- BB_DELINQUENCY_COUNT_MEAN
- BB_DELINQUENCY_COUNT_MAX
- BB_DELINQUENCY_COUNT_SUM
- BB_DELINQUENCY_RATE_MEAN
- BB_DELINQUENCY_RATE_MAX
- BB_STATUS_2_PLUS_RATE_MEAN
- BB_STATUS_2_PLUS_RATE_MAX
- BB_STATUS_1_COUNT_SUM
- BB_STATUS_2_PLUS_COUNT_SUM
- BB_STATUS_C_COUNT_SUM
- BB_STATUS_X_COUNT_SUM
- BB_EVER_DELINQUENT_SUM
- BB_EVER_SEVERE_DELINQUENCY_SUM
- BUREAU_CREDITS_WITH_BALANCE_HISTORY
- BUREAU_BALANCE_COVERAGE_RATIO

Total Bureau Balance features: 22


In [49]:
print("""
Model05 BUREAU BALANCE FEATURE ENGINEERING COMPLETE

Model04 benchmark: ROC-AUC = 0.777585, PR-AUC = 0.274161
Model05 result:    ROC-AUC = {:.4f}, PR-AUC = {:.4f}
Change:            ROC-AUC = {:+.4f}, PR-AUC = {:+.4f}

Long-term target: ~0.80 ROC-AUC

Next: POS_CASH_balance, installments_payments, credit_card_balance,
      then feature refinement, Optuna, SHAP.
""".format(roc_auc, pr_auc, roc_change, pr_change))


Model05 BUREAU BALANCE FEATURE ENGINEERING COMPLETE

Model04 benchmark: ROC-AUC = 0.777585, PR-AUC = 0.274161
Model05 result:    ROC-AUC = 0.7777, PR-AUC = 0.2745
Change:            ROC-AUC = +0.0001, PR-AUC = +0.0004

Long-term target: ~0.80 ROC-AUC

Next: POS_CASH_balance, installments_payments, credit_card_balance,
      then feature refinement, Optuna, SHAP.

